# PokerBench integration — safe, small-sample first

This is a **new notebook**; it does not modify the original Colab. The supplied Colab requires Google authentication from this environment, so its architecture was not inspected. Run the inspection cell in the original notebook first, then copy the appropriate integration cells below after its imports/configuration.


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: False
GPU: None


In [4]:
# Local dependency preflight. Install with: uv sync --all-extras
# Do not install or upgrade packages from inside the notebook: VS Code should use the project environment.
import importlib.util

required_modules = ('datasets', 'numpy', 'pandas', 'sklearn', 'torch', 'transformers')
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError(
        f'Missing notebook dependencies: {missing_modules}. '
        'From the repository root, run: uv sync --all-extras'
    )

import re
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
from datasets import load_dataset, get_dataset_split_names
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
DATASET_ID = "RZ412/PokerBench"
SMOKE_TRAIN_ROWS = 1_000
SMOKE_TEST_ROWS = 1_000
np.random.seed(SEED)

/Users/emilydu/Poker-Decision-Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Inspect the existing notebook before integrating

Run this in the original notebook—not as a substitute for manually reviewing its cells. Record the model's forward signature and expected feature shapes before choosing the text or structured adapter.


In [5]:
# ORIGINAL-NOTEBOOK INSPECTION (run after the original notebook defines its objects)
import inspect

print("torch available:", "torch" in globals() or __import__("importlib").util.find_spec("torch") is not None)
for name in ("model", "tokenizer", "train_loader", "train_dataset", "X_train"):
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")
        if name == "model":
            try:
                print("forward signature:", inspect.signature(obj.forward))
            except (TypeError, ValueError):
                pass

# Also inspect existing preprocessing cells for: text tokenizer/vectorizer, categorical encoders,
# numerical scalers, label classes, and the exact batch keys consumed by model.forward(...).


torch available: True


## 2. Load only a smoke-test subset

PokerBench's Hub view exposes `train` and `test` splits. The smoke loader requests at most 1,000 rows per split, preserving the source split rather than creating a new test set. Do not use `train_test_split` on the combined Hub dataset.


In [6]:
def load_pokerbench_smoke(train_rows=SMOKE_TRAIN_ROWS, test_rows=SMOKE_TEST_ROWS):
    splits = get_dataset_split_names(DATASET_ID)
    print("available source splits:", splits)
    if not {"train", "test"}.issubset(splits):
        raise ValueError(f"Expected train/test source splits, found {splits}")
    return {
        "train": load_dataset(DATASET_ID, split=f"train[:{train_rows}]"),
        "test": load_dataset(DATASET_ID, split=f"test[:{test_rows}]"),
    }

raw = load_pokerbench_smoke()
for split, ds in raw.items():
    print(f"{split}: shape={ds.shape}, columns={ds.column_names}")
    print(ds.select(range(min(5, len(ds)))).to_pandas()[["instruction", "output"]].to_string(index=False))


available source splits: ['train', 'test']
train: shape=(1000, 2), columns=['instruction', 'output']
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

## 3. Validate and normalize targets

The action is the classification target. For `bet N` and `raise N`, the numerical target is retained as a separate masked regression target; it is not silently discarded. The amount scaler is fit only on the training partition.


In [7]:
ACTION_RE = re.compile(r"^\s*(fold|check|call|bet|raise)(?:\s+(-?\d+(?:\.\d+)?))?\s*$", re.I)
ACTIONS = ["fold", "check", "call", "bet", "raise"]
ACTION_TO_ID = {action: i for i, action in enumerate(ACTIONS)}

def parse_output(value):
    if not isinstance(value, str):
        return None
    match = ACTION_RE.match(value)
    if not match:
        return None
    action, amount = match.groups()
    amount = None if amount is None else float(amount)
    # Amounts are meaningful only for bet/raise and must be positive.
    if amount is not None and (action.lower() not in {"bet", "raise"} or amount <= 0):
        return None
    return action.lower(), amount

def validate_and_prepare(dataset, split_name):
    frame = dataset.to_pandas().copy()
    required = {"instruction", "output"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{split_name} is missing {required - set(frame.columns)}")
    parsed = frame.output.map(parse_output)
    invalid = frame.instruction.isna() | frame.instruction.astype(str).str.strip().eq("") | parsed.isna()
    print(f"{split_name}: rows={len(frame)}, missing_by_column={frame.isna().sum().to_dict()}, invalid={int(invalid.sum())}")
    bad = frame.loc[invalid, ["instruction", "output"]]
    if len(bad):
        display(bad.head())
    frame = frame.loc[~invalid].reset_index(drop=True)
    frame[["action", "amount"]] = pd.DataFrame(parsed.loc[~invalid].tolist(), index=frame.index)
    frame["action_id"] = frame.action.map(ACTION_TO_ID).astype("int64")
    frame["has_amount"] = frame.amount.notna().astype("int64")
    print(f"{split_name} action distribution:\n{frame.action.value_counts(dropna=False)}")
    print(f"{split_name} amount-bearing targets={frame.has_amount.sum()}")
    return frame

train_frame = validate_and_prepare(raw["train"], "source-train")
test_frame = validate_and_prepare(raw["test"], "source-test")
assert set(train_frame.action).issubset(ACTIONS)
assert set(test_frame.action).issubset(ACTIONS)


source-train: rows=1000, missing_by_column={'instruction': 0, 'output': 0}, invalid=0
source-train action distribution:
action
check    264
fold     260
call     213
raise    154
bet      109
Name: count, dtype: int64
source-train amount-bearing targets=263
source-test: rows=1000, missing_by_column={'instruction': 0, 'output': 0}, invalid=0
source-test action distribution:
action
call     259
fold     238
check    234
raise    165
bet      104
Name: count, dtype: int64
source-test amount-bearing targets=269


In [8]:
# Development validation comes from SOURCE TRAIN ONLY. SOURCE TEST remains untouched until final evaluation.
train_idx, valid_idx = train_test_split(
    np.arange(len(train_frame)), test_size=0.2, random_state=SEED, stratify=train_frame["action"]
)
train_df = train_frame.iloc[train_idx].reset_index(drop=True)
valid_df = train_frame.iloc[valid_idx].reset_index(drop=True)
test_df = test_frame.reset_index(drop=True)

amount_scaler = StandardScaler()
train_amount_mask = train_df.has_amount.astype(bool)
amount_scaler.fit(train_df.loc[train_amount_mask, ["amount"]])

def add_scaled_amount(frame):
    result = frame.copy()
    result["amount_target"] = 0.0  # ignored where has_amount == 0
    mask = result.has_amount.astype(bool)
    result.loc[mask, "amount_target"] = amount_scaler.transform(result.loc[mask, ["amount"]]).ravel()
    return result

train_df, valid_df, test_df = map(add_scaled_amount, (train_df, valid_df, test_df))
print({name: len(frame) for name, frame in {"train": train_df, "valid": valid_df, "test": test_df}.items()})


{'train': 800, 'valid': 200, 'test': 1000}


## 4A. Text/Transformer adapter

Use this when the original model consumes token IDs or text features. It preserves the exact poker instruction as model input. The batch has action labels plus masked amount targets; an existing model must expose (or be wrapped to expose) an action-logit head and an amount head.


In [9]:
# Run only if the original notebook is text/Transformer-based.
import torch
from torch.utils.data import Dataset, DataLoader

class PokerTextDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=512):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        encoded = self.tokenizer(
            row.instruction, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        return {
            "model_inputs": {key: value.squeeze(0) for key, value in encoded.items()},
            "action_id": torch.tensor(row.action_id, dtype=torch.long),
            "amount_target": torch.tensor(row.amount_target, dtype=torch.float32),
            "has_amount": torch.tensor(row.has_amount, dtype=torch.bool),
            "raw_output": row.output,
        }

def poker_collate(batch):
    return {
        "model_inputs": {key: torch.stack([x["model_inputs"][key] for x in batch]) for key in batch[0]["model_inputs"]},
        "action_id": torch.stack([x["action_id"] for x in batch]),
        "amount_target": torch.stack([x["amount_target"] for x in batch]),
        "has_amount": torch.stack([x["has_amount"] for x in batch]),
        "raw_output": [x["raw_output"] for x in batch],
    }

# Reuse the existing notebook's tokenizer. Do NOT refit it on test data.
# train_loader = DataLoader(PokerTextDataset(train_df, tokenizer), batch_size=8, shuffle=True, collate_fn=poker_collate)
# valid_loader = DataLoader(PokerTextDataset(valid_df, tokenizer), batch_size=8, shuffle=False, collate_fn=poker_collate)
# test_loader  = DataLoader(PokerTextDataset(test_df, tokenizer), batch_size=8, shuffle=False, collate_fn=poker_collate)


## 4B. Structured CSV adapter (optional and intentionally not downloaded by default)

The Hugging Face dataset provides the JSON instruction/output pairs for the primary pipeline. Download a named CSV only if the existing model expects structured numeric/categorical features. The pre-flop and post-flop CSV schemas differ, so preprocess them independently and fit every encoder/scaler on the corresponding training CSV only.


In [10]:

# Optional: fetch exactly one documented CSV; this avoids downloading the full corpus by accident.
# from huggingface_hub import hf_hub_download
# PREFLOP_TRAIN_CSV = "preflop_60k_train_set_game_scenario_information.csv"
# PREFLOP_TEST_CSV = "preflop_1k_test_set_game_scenario_information.csv"
# path = hf_hub_download(repo_id=DATASET_ID, repo_type="dataset", filename=PREFLOP_TRAIN_CSV)
# preflop_train = pd.read_csv(path)
# print(preflop_train.shape, preflop_train.columns.tolist())
# display(preflop_train.head())
#
# Do not concatenate pre-flop and post-flop CSVs: their column schemas differ.
# Do not fit encoders/scalers on *_test.csv. Parse available_moves and preserve correct_decision amounts.


## 5. One-batch contract test and prediction decoding

This deliberately does not invent a replacement model. Set `forward_adapter` to match the inspected model's forward contract, then run the test. It verifies a batch, expected output dimensions, a masked loss, one optimizer step, and decoded action/amount predictions.


In [11]:
import torch.nn.functional as F

def decode_prediction(action_id, scaled_amount=None):
    action = ACTIONS[int(action_id)]
    if action in {"bet", "raise"} and scaled_amount is not None:
        amount = amount_scaler.inverse_transform(np.array([[float(scaled_amount)]])).item()
        return f"{action} {amount:.2f}"
    return action

def one_step_smoke_test(model, optimizer, loader, forward_adapter, device="cpu"):
    """forward_adapter(model, model_inputs) -> (action_logits[B, 5], amount_prediction[B])."""
    model.train()
    batch = next(iter(loader))
    inputs = {key: value.to(device) for key, value in batch["model_inputs"].items()}
    action_target = batch["action_id"].to(device)
    amount_target = batch["amount_target"].to(device)
    amount_mask = batch["has_amount"].to(device)
    action_logits, amount_pred = forward_adapter(model, inputs)
    assert action_logits.shape == (len(action_target), len(ACTIONS)), action_logits.shape
    assert amount_pred.shape == amount_target.shape, amount_pred.shape
    action_loss = F.cross_entropy(action_logits, action_target)
    amount_loss = F.mse_loss(amount_pred[amount_mask], amount_target[amount_mask]) if amount_mask.any() else 0.0
    loss = action_loss + amount_loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    predicted = action_logits.detach().argmax(dim=-1).cpu().numpy()
    decoded = [decode_prediction(a, m) for a, m in zip(predicted, amount_pred.detach().cpu().numpy())]
    print({"loss": float(loss.detach()), "decoded_predictions": decoded[:5], "gold_outputs": batch["raw_output"][:5]})
    return batch, decoded

# Example adapter only if the inspected model already returns a mapping with these keys:
# forward_adapter = lambda model, inputs: (model(**inputs)["action_logits"], model(**inputs)["amount_prediction"].squeeze(-1))
# batch, decoded = one_step_smoke_test(model, optimizer, train_loader, forward_adapter, device)


## Scale safely

After the smoke test succeeds, increase `train[:1000]` to `train` and `test[:1000]` to `test` one split at a time. For memory-sensitive experiments, use `load_dataset(..., streaming=True)` and deterministic `.take(n)` for inspection, then a streaming/iterable dataset for training. Keep the official test split fixed, fit preprocessing only on source-train, and make validation only from source-train.


In [12]:
# ===== BASELINES (uses train_df / valid_df / test_df from the split cell) =====
import numpy as np, pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, f1_score

def score(name, y_true, proba, classes):
    pred = classes[proba.argmax(1)]
    return {"model": name,
            "accuracy": accuracy_score(y_true, pred),
            "log_loss": log_loss(y_true, proba, labels=classes),
            "macro_F1": f1_score(y_true, pred, average="macro")}

def run_baselines(tr, ev):
    rows = []
    # 1) Majority / class prior: the floor
    dummy = DummyClassifier(strategy="prior").fit(np.zeros((len(tr), 1)), tr.action)
    rows.append(score("majority (prior)", ev.action,
                      dummy.predict_proba(np.zeros((len(ev), 1))), dummy.classes_))
    # 2) TF-IDF on the instruction text + logistic regression
    text = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                         LogisticRegression(max_iter=2000))
    text.fit(tr.instruction, tr.action)
    rows.append(score("tfidf + logreg", ev.action,
                      text.predict_proba(ev.instruction), text.classes_))
    return pd.DataFrame(rows).round(3)

# Develop on validation only. Do NOT score test_df until the final evaluation.
baseline_valid = run_baselines(train_df, valid_df)
display(baseline_valid)


,model,accuracy,log_loss,macro_F1
0,majority (prior),0.265,1.562,0.084
1,tfidf + logreg,0.540,1.243,0.420


In [13]:

import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "google/bert_uncased_L-4_H-256_A-4"
MAX_LENGTH = 512
BATCH_SIZE = 3

if len(train_df) < BATCH_SIZE:
    raise ValueError(f"Expected at least {BATCH_SIZE} training examples, found {len(train_df)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"Device: {device} ({gpu_name})")

smoke_texts = train_df["instruction"].iloc[:BATCH_SIZE].astype(str).tolist()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_mini = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_mini.eval()

encoded = tokenizer(
    smoke_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_attention_mask=True,
    return_tensors="pt",
)
model_inputs = {name: tensor.to(device) for name, tensor in encoded.items()}

with torch.no_grad():
    outputs = bert_mini(**model_inputs)

token_embeddings = outputs.last_hidden_state
cls_embeddings = token_embeddings[:, 0, :]
all_finite = bool(
    torch.isfinite(token_embeddings).all()
    and torch.isfinite(cls_embeddings).all()
)

print("Input token IDs shape:", tuple(model_inputs["input_ids"].shape))
print("Attention mask shape:", tuple(model_inputs["attention_mask"].shape))
print("Token embeddings shape:", tuple(token_embeddings.shape))
print("CLS embeddings shape:", tuple(cls_embeddings.shape))
print("Embedding device:", token_embeddings.device)
print("All embeddings finite:", all_finite)

assert token_embeddings.shape[0] == BATCH_SIZE, token_embeddings.shape
assert token_embeddings.shape[2] == 256, token_embeddings.shape
assert cls_embeddings.shape == (BATCH_SIZE, 256), cls_embeddings.shape
assert all_finite, "BERT Mini produced non-finite embeddings"
assert token_embeddings.device.type == device.type
assert cls_embeddings.device.type == device.type

print("SUCCESS: BERT Mini smoke test passed!")

Device: cpu (CPU)
Input token IDs shape: (3, 266)
Attention mask shape: (3, 266)
Token embeddings shape: (3, 266, 256)
CLS embeddings shape: (3, 256)
Embedding device: cpu
All embeddings finite: True
SUCCESS: BERT Mini smoke test passed!


## Meeting 2 — Structured PokerBench Candidate Decision Model

This section uses the team pipeline at commit `b4a42dd42ef5537ad4a75b8f441e3cae61c57009`. It reads only pinned PokerBench training CSVs, preserves the team's answer-free `DecisionInputs(context, candidates)` contract, and never loads the official test set. The existing exploratory pipeline, baseline results, and BERT Mini smoke test above are unchanged.


In [14]:
# Local repository setup. Run this notebook through the project's uv environment in VS Code.
# The same cell also works in Colab after opening/cloning this repository locally there.
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src/poker_decision').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not find the Poker-Decision-Model checkout from the notebook working directory. '
        'Open the notebook from a local repository checkout.'
    )

ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
assert (ROOT / 'src/poker_decision/pipeline.py').is_file()
try:
    package_version = version('poker-decision-model')
except PackageNotFoundError:
    package_version = 'not installed (using local src path; prefer uv sync --all-extras)'
print('Repository:', ROOT)
print('Package:', package_version)


Repository: /Users/emilydu/Poker-Decision-Model
Package: 0.1.0


In [15]:
import sys

sys.path.insert(0, "/content/Poker-Decision-Model/src")

import poker_decision

print("poker_decision loaded!")
print(poker_decision.__file__)

poker_decision loaded!
/Users/emilydu/Poker-Decision-Model/src/poker_decision/__init__.py


### 1. Validated candidate data and context-disjoint split

`prepare()` verifies pinned raw training CSV checksums, parses `available_moves` into legal candidate menus, requires the separate `correct_decision` to match exactly one candidate, and stops on every parse failure. It then splits by answer-free context.


In [16]:
import json
import pandas as pd

from poker_decision.data import TRAIN_FILES, build_inputs, read_rows
from poker_decision.pipeline import prepare

MEETING2_SEED = 42
SAMPLE_PER_KIND = 5_000  # Every source row is audited before this reproducible working sample.
train_examples, validation_examples, manifest = prepare(
    ROOT, sample_per_kind=SAMPLE_PER_KIND, seed=MEETING2_SEED, validation_fraction=0.20
)
assert manifest['official_test_used'] is False
assert {e.inputs.context for e in train_examples}.isdisjoint(e.inputs.context for e in validation_examples)

def examples_to_frame(examples):
    rows = []
    for e in examples:
        assert set(e.inputs.__dataclass_fields__) == {'context', 'candidates'}
        assert len(set(e.inputs.candidates)) == len(e.inputs.candidates)
        assert 0 <= e.target_index < len(e.inputs.candidates)
        rows.append({'source_id': e.source_id, 'street': e.street, 'context': e.inputs.context,
                     'candidates': list(e.inputs.candidates), 'target_index': e.target_index,
                     'target_action': e.target_action})
    return pd.DataFrame(rows)

candidate_train_df = examples_to_frame(train_examples)
candidate_validation_df = examples_to_frame(validation_examples)
for name, frame in {'train': candidate_train_df, 'validation': candidate_validation_df}.items():
    assert frame.apply(lambda r: 0 <= r.target_index < len(r.candidates), axis=1).all()
    assert frame.apply(lambda r: r.candidates[r.target_index] == r.target_action, axis=1).all()
    print(f'{name}: {len(frame):,} validated decision examples')

# The parser's allowlist makes the input unchanged if answer fields are removed or added.
for kind, filename in TRAIN_FILES.items():
    raw_row = read_rows(ROOT / 'data/raw/pokerbench' / filename)[0]
    original_inputs = build_inputs(raw_row, kind)
    raw_row.pop('correct_decision')
    raw_row['answer'] = 'THIS_MUST_NOT_ENTER_THE_INPUT'
    assert build_inputs(raw_row, kind) == original_inputs

print('Official test used:', manifest['official_test_used'])
display(candidate_train_df[['source_id', 'street', 'candidates', 'target_index', 'target_action']].head(3))


preflop: 63,200 source rows; 63,200 valid; 0 failures
postflop: 500,000 source rows; 500,000 valid; 0 failures
train: 8,000 validated decision examples
validation: 2,000 validated decision examples
Official test used: False


,source_id,street,candidates,target_index,target_action
0,preflop-train-41905,preflop,"[call, fold]",0,call
1,preflop-train-48598,preflop,"[call, fold]",0,call
2,preflop-train-18024,preflop,"[call, fold]",1,fold


### 2. Candidate spans, BERT Mini, and masked shared scoring

Each decision is serialized into one sequence: state, question, and every supplied candidate. Candidate token spans are tracked exactly. Inputs longer than 512 tokens fail rather than being silently truncated. The shared linear head scores mean-pooled contextual embeddings for each candidate; padding is masked before softmax.


In [17]:
import torch
from poker_decision.models.encoder import BACKBONES, load_pretrained
from poker_decision.training import batch_on_device, encode_examples

BACKBONE = 'bert_mini'
MODEL_NAME, MODEL_REVISION = BACKBONES[BACKBONE]
MAX_LENGTH = 512
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

meeting2_tokenizer, decision_model = load_pretrained(MODEL_NAME, MODEL_REVISION)
decision_model = decision_model.to(DEVICE).eval()
MAX_LENGTH = min(MAX_LENGTH, int(decision_model.encoder.config.max_position_embeddings))
train_encoded = encode_examples(train_examples, meeting2_tokenizer, max_length=MAX_LENGTH)
validation_encoded = encode_examples(validation_examples, meeting2_tokenizer, max_length=MAX_LENGTH)
PAD = meeting2_tokenizer.pad_token_id
assert decision_model.encoder.config.hidden_size == 256
assert len(train_encoded) == len(train_examples) and len(validation_encoded) == len(validation_examples)

forward_batch = batch_on_device(train_encoded[:3], PAD, DEVICE)
with torch.no_grad():
    candidate_logits = decision_model(**forward_batch)
    candidate_probabilities = candidate_logits.softmax(dim=-1)

assert torch.allclose(candidate_probabilities.sum(dim=-1), torch.ones(3, device=DEVICE), atol=1e-6)
assert torch.isfinite(candidate_logits[torch.isfinite(candidate_logits)]).all()
print('Device:', DEVICE)
print('Tokenization passed without truncation; max length:', MAX_LENGTH)
print('Candidate logits shape:', tuple(candidate_logits.shape))
print('Candidate probabilities sum to:', candidate_probabilities.sum(dim=-1).detach().cpu().tolist())


Device: cpu
Tokenization passed without truncation; max length: 512
Candidate logits shape: (3, 2)
Candidate probabilities sum to: [1.0, 1.0, 1.0]


### 3. Tiny-batch overfitting sanity check

This trains a fresh BERT Mini plus shared head on 32 distinct, multi-option training contexts only. A high training accuracy is a wiring check, not a validation result.


In [18]:
import gc
import random
from transformers import AutoModel
from poker_decision.models.encoder import PokerDecisionModel
from poker_decision.training import score_model, seed_everything, train_epoch

TINY_N, TINY_EPOCHS = 32, 100
seen_contexts, tiny_examples = set(), []
for example in random.Random(MEETING2_SEED).sample(train_examples, len(train_examples)):
    if len(example.inputs.candidates) > 1 and example.inputs.context not in seen_contexts:
        tiny_examples.append(example)
        seen_contexts.add(example.inputs.context)
    if len(tiny_examples) == TINY_N:
        break
if len(tiny_examples) != TINY_N:
    raise ValueError('Need 32 distinct multi-option training contexts')
tiny_encoded = encode_examples(tiny_examples, meeting2_tokenizer, max_length=MAX_LENGTH)

seed_everything(MEETING2_SEED)
tiny_model = PokerDecisionModel(AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)).to(DEVICE)
tiny_optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=1e-4, weight_decay=0.0)
tiny_history = []
for epoch in range(1, TINY_EPOCHS + 1):
    train_loss = train_epoch(tiny_model, tiny_encoded, [e.target_index for e in tiny_examples],
                             pad_token_id=PAD, optimizer=tiny_optimizer, batch_size=8,
                             seed=MEETING2_SEED + epoch)
    metrics = score_model(tiny_model, tiny_examples, tiny_encoded, pad_token_id=PAD)['overall']
    tiny_history.append({'epoch': epoch, 'train_loss': train_loss, **metrics})
    if epoch == 1 or epoch % 10 == 0 or metrics['exact_accuracy'] == 1:
        print(f"Tiny epoch {epoch:03d}: loss={train_loss:.4f}, accuracy={metrics['exact_accuracy']:.1%}")
    if metrics['exact_accuracy'] == 1:
        break
display(pd.DataFrame(tiny_history))
if tiny_history[-1]['exact_accuracy'] < 0.95:
    raise RuntimeError('Tiny learning check failed; inspect inputs, labels, loss, and gradients.')
print('SUCCESS: tiny candidate-ranking overfit test passed.')
del tiny_model, tiny_optimizer
gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()


Tiny epoch 001: loss=0.9375, accuracy=65.6%
Tiny epoch 008: loss=0.2786, accuracy=100.0%


,epoch,train_loss,n,exact_accuracy,tie_expected_accuracy,action_type_macro_f1,log_loss,brier_score,single_option_rate,multi_option_exact_accuracy
0,1,0.937485,32.0,0.65625,0.65625,0.581019,0.745379,0.464060,0.0,0.65625
1,2,0.726034,32.0,0.65625,0.65625,0.580745,0.686639,0.427110,0.0,0.65625
2,3,0.714960,32.0,0.68750,0.68750,0.601852,0.629533,0.394102,0.0,0.68750
3,4,0.612748,32.0,0.75000,0.75000,0.647619,0.525309,0.316805,0.0,0.75000
4,5,0.529724,32.0,0.90625,0.90625,0.919495,0.411397,0.231382,0.0,0.90625
5,6,0.489190,32.0,0.90625,0.90625,0.919495,0.331075,0.177809,0.0,0.90625
6,7,0.416552,32.0,0.93750,0.93750,0.956439,0.227683,0.112670,0.0,0.93750
7,8,0.278641,32.0,1.00000,1.00000,1.000000,0.151336,0.066044,0.0,1.00000


SUCCESS: tiny candidate-ranking overfit test passed.


### 4. Full training, validation comparison, and checkpoint

Both BERT Mini and its shared candidate head are fine-tuned. Training uses only the prepared training partition; validation uses the same saved context-disjoint partition as Uniform, Menu Prior, and Linear Candidate Scorer.


In [19]:
import hashlib
from datetime import UTC, datetime
from transformers import AutoModel
from poker_decision.evaluation import evaluate_by_street
from poker_decision.models.encoder import PokerDecisionModel
from poker_decision.pipeline import run_baselines, write_json, write_jsonl
from poker_decision.training import prediction_lookup, predict_encoded, save_checkpoint

EPOCHS, BATCH_SIZE = 3, 16
RUN_ID = datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
OUT = ROOT / 'artifacts/pokerbench_meeting2' / f'bert-mini-{RUN_ID}'
OUT.mkdir(parents=True, exist_ok=True)
write_json(OUT / 'data_manifest.json', manifest)
write_jsonl(OUT / 'train.jsonl', [e.to_record() for e in train_examples])
write_jsonl(OUT / 'validation.jsonl', [e.to_record() for e in validation_examples])

seed_everything(MEETING2_SEED)
model = PokerDecisionModel(AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)).to(DEVICE)
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 2e-5},
    {'params': model.head.parameters(), 'lr': 1e-3},
], weight_decay=0.01)

history, best_log_loss = [], float('inf')
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_encoded, [e.target_index for e in train_examples],
                             pad_token_id=PAD, optimizer=optimizer, batch_size=BATCH_SIZE,
                             seed=MEETING2_SEED + epoch)
    validation_metrics = score_model(model, validation_examples, validation_encoded, pad_token_id=PAD)['overall']
    row = {'epoch': epoch, 'train_loss': train_loss, **validation_metrics}
    history.append(row)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_accuracy={validation_metrics['exact_accuracy']:.1%}, val_log_loss={validation_metrics['log_loss']:.4f}")
    write_json(OUT / 'training_history.json', history)
    if validation_metrics['log_loss'] < best_log_loss:
        best_log_loss = validation_metrics['log_loss']
        save_checkpoint(OUT / 'best', model, meeting2_tokenizer, {'best_epoch': epoch, 'validation_log_loss': best_log_loss, 'official_test_used': False, 'backbone': MODEL_NAME, 'revision': MODEL_REVISION})

display(pd.DataFrame(history))
model_probabilities = predict_encoded(model, validation_encoded, pad_token_id=PAD)
model_lookup = prediction_lookup(validation_examples, model_probabilities)
baseline_report = run_baselines(OUT, train_examples, validation_examples, seed=MEETING2_SEED)
comparison = dict(baseline_report['metrics'])
comparison['bert_mini_shared_scorer'] = evaluate_by_street(validation_examples, model_lookup.__getitem__)
comparison_table = pd.DataFrame([
    {'model': name, 'subset': subset, **metrics}
    for name, subsets in comparison.items() for subset, metrics in subsets.items()
]).set_index(['model', 'subset'])
write_json(OUT / 'comparison.json', comparison)
display(comparison_table.round(4))
print('Checkpoint and reports:', OUT)


Epoch 1: train_loss=0.7706, val_accuracy=60.6%, val_log_loss=0.7246
Epoch 2: train_loss=0.7073, val_accuracy=70.3%, val_log_loss=0.6430
Epoch 3: train_loss=0.6563, val_accuracy=72.0%, val_log_loss=0.6173


,epoch,train_loss,n,exact_accuracy,tie_expected_accuracy,action_type_macro_f1,log_loss,brier_score,single_option_rate,multi_option_exact_accuracy
0,1,0.770551,2000.0,0.6055,0.6055,0.507393,0.724571,0.469215,0.0485,0.585391
1,2,0.707311,2000.0,0.7035,0.7035,0.574517,0.643020,0.397403,0.0485,0.688387
2,3,0.656284,2000.0,0.7205,0.7205,0.617421,0.617322,0.372030,0.0485,0.706253


{
  "uniform": {
    "n": 2000.0,
    "exact_accuracy": 0.431,
    "tie_expected_accuracy": 0.451,
    "action_type_macro_f1": 0.2475685011973849,
    "log_loss": 0.8435929740857674,
    "brier_score": 0.549,
    "single_option_rate": 0.04849999999999999,
    "multi_option_exact_accuracy": 0.40199684708355227
  },
  "option_prior": {
    "n": 2000.0,
    "exact_accuracy": 0.59,
    "tie_expected_accuracy": 0.59,
    "action_type_macro_f1": 0.47047321867440467,
    "log_loss": 0.7462330194219485,
    "brier_score": 0.483115861298033,
    "single_option_rate": 0.04849999999999999,
    "multi_option_exact_accuracy": 0.5691014188124015
  },
  "linear_candidate": {
    "n": 2000.0,
    "exact_accuracy": 0.729,
    "tie_expected_accuracy": 0.729,
    "action_type_macro_f1": 0.6503763728414885,
    "log_loss": 0.7919247479631577,
    "brier_score": 0.4099180339169037,
    "single_option_rate": 0.04849999999999999,
    "multi_option_exact_accuracy": 0.7151865475564898
  }
}


n  exact_accuracy  \
model                   subset                            
uniform                 overall  2000.0          0.4310   
                        flop       72.0          0.7083   
                        preflop  1027.0          0.4869   
                        river     499.0          0.3307   
                        turn      402.0          0.3632   
option_prior            overall  2000.0          0.5900   
                        flop       72.0          0.5556   
                        preflop  1027.0          0.6680   
                        river     499.0          0.5110   
                        turn      402.0          0.4950   
linear_candidate        overall  2000.0          0.7290   
                        flop       72.0          0.5278   
                        preflop  1027.0          0.8832   
                        river     499.0          0.5471   
                        turn      402.0          0.5970   
bert_mini_shared_scorer overall  2000.0          0.7205   
                        flop       72.0          0.4306   
                        preflop  1027.0          0.8500   
                        river     499.0          0.5852   
                        turn      402.0          0.6095   

                                 tie_expected_accuracy  action_type_macro_f1  \
model                   subset                                                 
uniform                 overall                 0.4510                0.2476   
                        flop                    0.3507                0.3256   
                        preflop                 0.5140                0.3362   
                        river                   0.4200                0.2169   
                        turn                    0.3464                0.1958   
option_prior            overall                 0.5900                0.4705   
                        flop                    0.5556                0.2850   
                        preflop                 0.6680                0.6418   
                        river                   0.5110                0.3221   
                        turn                    0.4950                0.3338   
linear_candidate        overall                 0.7290                0.6504   
                        flop                    0.5278                0.3683   
                        preflop                 0.8832                0.9074   
                        river                   0.5471                0.4775   
                        turn                    0.5970                0.5405   
bert_mini_shared_scorer overall                 0.7205                0.6174   
                        flop                    0.4306                0.3370   
                        preflop                 0.8500                0.8859   
                        river                   0.5852                0.4120   
                        turn                    0.6095                0.5632   

                                 log_loss  brier_score  single_option_rate  \
model                   subset                                               
uniform                 overall    0.8436       0.5490              0.0485   
                        flop       1.0694       0.6493              0.0000   
                        preflop    0.7121       0.4860              0.0944   
                        river      0.8930       0.5800              0.0000   
                        turn       1.0776       0.6536              0.0000   
option_prior            overall    0.7462       0.4831              0.0485   
                        flop       1.0377       0.6427              0.0000   
                        preflop    0.5953       0.4096              0.0944   
                        river      0.8403       0.5374              0.0000   
                        turn       0.9629       0.5750              0.0000   
linear_candidate        overall    0.7919 

Checkpoint and reports: /Users/emilydu/Poker-Decision-Model/artifacts/pokerbench_meeting2/bert-mini-20261029T024813Z


### 5. Five validation mistakes

Inspecting individual wrong predictions helps distinguish action-type confusion from exact sizing errors. These are validation examples only.


In [20]:
from poker_decision.evaluation import inspect_mistakes

mistakes = inspect_mistakes(validation_examples, model_lookup.__getitem__, seed=MEETING2_SEED)
write_jsonl(OUT / 'five_errors.jsonl', mistakes)
print(f'Showing {len(mistakes)} validation mistakes')
for number, mistake in enumerate(mistakes, start=1):
    print(f"\nMistake {number}: {mistake['source_id']}")
    print(mistake['context'])
    print('Reference:', mistake['target_action'], '| Prediction:', mistake['predicted_action'])
    display(pd.DataFrame({'candidate': mistake['candidates'], 'probability': mistake['probabilities']}))


Showing 5 validation mistakes

Mistake 1: preflop-train-12109
Street: preflop. Hero position: BB. Hero cards: Ks Th. Previous actions: BTN: raise 2.5bb; SB: call; BB: raise 11bb; BTN: call; SB: all-in. Source num_players: 3. Source num_bets: 3. Pot: 122 bb.
Reference: fold | Prediction: call


,candidate,probability
0,call,0.509838
1,fold,0.490162



Mistake 2: preflop-train-35272
Street: preflop. Hero position: BB. Hero cards: Ad Qc. Previous actions: UTG: raise 2bb; CO: call; SB: call; BB: raise 14bb; UTG: call; CO: fold; SB: all-in. Source num_players: 4. Source num_bets: 3. Pot: 130 bb.
Reference: fold | Prediction: call


,candidate,probability
0,call,0.872514
1,fold,0.127486



Mistake 3: postflop-train-266795
Street: river. Hero position: IP. Hero cards: Ac 9d. Board: 9s 7s 4c Ah 8c. Preflop actions: SB: raise 3bb; BB: call. Postflop actions: OOP: check; IP: bet 5bb; OOP: call; turn: Ah; OOP: check; IP: bet 12bb; OOP: raise 38bb; IP: call; river: 8c; OOP: check. Pot: 87 bb.
Reference: raise 54bb | Prediction: check


,candidate,probability
0,check,0.777552
1,raise 54bb,0.222448



Mistake 4: postflop-train-100168
Street: river. Hero position: OOP. Hero cards: Js 5s. Board: Jc Jh 4s 4d As. Preflop actions: SB: raise 3bb; BB: call. Postflop actions: OOP: check; IP: check; turn: 4d; OOP: check; IP: check; river: As; OOP: bet 3bb; IP: raise 11bb. Pot: 20 bb.
Reference: raise 29bb | Prediction: fold


,candidate,probability
0,fold,0.760268
1,call,0.158492
2,raise 29bb,0.081240



Mistake 5: postflop-train-449174
Street: river. Hero position: OOP. Hero cards: Th 8h. Board: Jh 4h 5s Kh 6h. Preflop actions: HJ: raise 2bb; BTN: call. Postflop actions: OOP: check; IP: bet 4bb; OOP: call; turn: Kh; OOP: check; IP: bet 16bb; OOP: call; river: 6h; OOP: bet 23bb; IP: raise 78bb. Pot: 142 bb.
Reference: call | Prediction: fold


,candidate,probability
0,fold,0.680023
1,call,0.319977


## Independent Meeting 2 analysis — candidate menus versus poker state

This analysis is intentionally separate from the team's canonical Meeting 2 implementation. It **does not train a model or alter the validation split**. It reloads the canonical best BERT Mini checkpoint and evaluates the original 2,000 validation examples alongside a seed-42 state-shuffled copy that retains each example's identity, candidate menu, and label.

Interpret the shuffle diagnostic as sensitivity to this dataset's serialized state/context under semantically mismatched state–menu pairs; it is not, by itself, a test of poker understanding.


In [21]:
# CHECKPOINT-ONLY ANALYSIS: no fitting, fine-tuning, or mutation of validation_examples.
# Run after the canonical Meeting 2 training/evaluation cells in the same local or Colab runtime.
import random
from collections import Counter

import pandas as pd

from poker_decision.data import DecisionExample, DecisionInputs
from poker_decision.evaluation import evaluate, evaluate_by_street
from poker_decision.training import (
    encode_examples,
    load_checkpoint,
    predict_encoded,
    prediction_lookup,
)

ANALYSIS_SEED = 42
CHECKPOINT = OUT / 'best'
if not CHECKPOINT.is_dir():
    raise FileNotFoundError(
        f'Missing canonical BERT Mini checkpoint: {CHECKPOINT}. '
        'Run the canonical training cell first; this analysis never trains a replacement model.'
    )
if len(validation_examples) != 2_000:
    raise ValueError(f'Expected the canonical 2,000-example validation split, got {len(validation_examples)}')

# Loading the saved best checkpoint makes this independent of the live in-memory training model.
analysis_tokenizer, analysis_model, analysis_metadata = load_checkpoint(CHECKPOINT)
if analysis_metadata.get('backbone') != MODEL_NAME or analysis_metadata.get('revision') != MODEL_REVISION:
    raise ValueError('Checkpoint backbone/revision does not match the canonical BERT Mini run')
analysis_model = analysis_model.to(DEVICE).eval()
analysis_max_length = min(MAX_LENGTH, int(analysis_model.encoder.config.max_position_embeddings))
analysis_pad = analysis_tokenizer.pad_token_id
if analysis_pad is None:
    raise ValueError('Checkpoint tokenizer has no pad token')

# Re-encode through the canonical serializer/candidate-span/masking path.
original_encoded = encode_examples(
    validation_examples, analysis_tokenizer, max_length=analysis_max_length
)
original_probabilities = predict_encoded(
    analysis_model, original_encoded, pad_token_id=analysis_pad
)
original_lookup = prediction_lookup(validation_examples, original_probabilities)
original_metrics = evaluate_by_street(validation_examples, original_lookup.__getitem__)

assert len(original_probabilities) == len(validation_examples) == 2_000
assert all(len(p) == len(e.inputs.candidates) for e, p in zip(validation_examples, original_probabilities))
print(f"Reloaded checkpoint best epoch: {analysis_metadata['best_epoch']}; original accuracy: {original_metrics['overall']['exact_accuracy']:.2%}")


Reloaded checkpoint best epoch: 3; original accuracy: 72.05%


In [22]:
# 1. Candidate-count analysis (BERT Mini only; no baseline is re-fit for this diagnostic).
def candidate_count_bucket(count):
    return '5+' if count >= 5 else str(count)

candidate_counts = Counter(len(e.inputs.candidates) for e in validation_examples)
candidate_count_distribution = pd.DataFrame(
    [
        {'candidate_count': count, 'examples': n, 'share_of_validation': n / len(validation_examples)}
        for count, n in sorted(candidate_counts.items())
    ]
)
display(candidate_count_distribution.style.format({'share_of_validation': '{:.1%}'}))

rows = []
for bucket in sorted({candidate_count_bucket(n) for n in candidate_counts}, key=lambda x: (x == '5+', int(x.rstrip('+')))):
    subset = [e for e in validation_examples if candidate_count_bucket(len(e.inputs.candidates)) == bucket]
    metrics = evaluate(subset, original_lookup.__getitem__)
    rows.append({
        'candidate_count': bucket,
        'examples': len(subset),
        'bert_mini_exact_accuracy': metrics['exact_accuracy'],
        'expected_uniform_accuracy': sum(1 / len(e.inputs.candidates) for e in subset) / len(subset),
    })
candidate_count_report = pd.DataFrame(rows)
display(candidate_count_report.style.format({
    'bert_mini_exact_accuracy': '{:.2%}',
    'expected_uniform_accuracy': '{:.2%}',
}))
print(
    'Linear candidate accuracy is intentionally omitted here: the canonical report retains aggregate '
    'metrics, not per-example predictions. Re-fitting it would be an additional training run.'
)


,candidate_count,examples,share_of_validation
0,1,97,4.9%
1,2,1093,54.6%
2,3,672,33.6%
3,4,138,6.9%


,candidate_count,examples,bert_mini_exact_accuracy,expected_uniform_accuracy
0,1,97,100.00%,100.00%
1,2,1093,79.87%,50.00%
2,3,672,59.38%,33.33%
3,4,138,52.17%,25.00%


Linear candidate accuracy is intentionally omitted here: the canonical report retains aggregate metrics, not per-example predictions. Re-fitting it would be an additional training run.


In [23]:
# 2. Seed-42 deranged state shuffle: menus, labels, streets, and source IDs stay at their original indices.
state_donors = list(range(len(validation_examples)))
shuffle_rng = random.Random(ANALYSIS_SEED)
while True:
    shuffle_rng.shuffle(state_donors)
    if all(recipient != donor for recipient, donor in enumerate(state_donors)):
        break

shuffled_validation_examples = [
    DecisionExample(
        source_id=example.source_id,
        street=example.street,
        inputs=DecisionInputs(
            context=validation_examples[donor].inputs.context,
            candidates=example.inputs.candidates,
        ),
        target_index=example.target_index,
    )
    for example, donor in zip(validation_examples, state_donors, strict=True)
]

# Guardrails: no mutation/reordering of the original set, and only context is changed.
assert sorted(state_donors) == list(range(len(validation_examples)))
assert all(recipient != donor for recipient, donor in enumerate(state_donors))
assert [e.source_id for e in shuffled_validation_examples] == [e.source_id for e in validation_examples]
assert [e.street for e in shuffled_validation_examples] == [e.street for e in validation_examples]
assert [e.inputs.candidates for e in shuffled_validation_examples] == [e.inputs.candidates for e in validation_examples]
assert [e.target_index for e in shuffled_validation_examples] == [e.target_index for e in validation_examples]
assert [e.target_action for e in shuffled_validation_examples] == [e.target_action for e in validation_examples]
assert any(
    original.inputs.context != shuffled.inputs.context
    for original, shuffled in zip(validation_examples, shuffled_validation_examples, strict=True)
)

shuffled_encoded = encode_examples(
    shuffled_validation_examples, analysis_tokenizer, max_length=analysis_max_length
)
shuffled_probabilities = predict_encoded(
    analysis_model, shuffled_encoded, pad_token_id=analysis_pad
)
shuffled_lookup = prediction_lookup(shuffled_validation_examples, shuffled_probabilities)
shuffled_metrics = evaluate_by_street(
    shuffled_validation_examples, shuffled_lookup.__getitem__
)

overall_shuffle_report = pd.DataFrame([
    {
        'condition': 'original state',
        'accuracy': original_metrics['overall']['exact_accuracy'],
        'log_loss': original_metrics['overall']['log_loss'],
    },
    {
        'condition': 'shuffled state',
        'accuracy': shuffled_metrics['overall']['exact_accuracy'],
        'log_loss': shuffled_metrics['overall']['log_loss'],
    },
    {
        'condition': 'original minus shuffled',
        'accuracy': original_metrics['overall']['exact_accuracy'] - shuffled_metrics['overall']['exact_accuracy'],
        'log_loss': shuffled_metrics['overall']['log_loss'] - original_metrics['overall']['log_loss'],
    },
])
display(overall_shuffle_report.style.format({'accuracy': '{:.2%}', 'log_loss': '{:.4f}'}))
print(
    f"Accuracy drop: {100 * (original_metrics['overall']['exact_accuracy'] - shuffled_metrics['overall']['exact_accuracy']):.2f} percentage points"
)


,condition,accuracy,log_loss
0,original state,72.05%,0.6173
1,shuffled state,54.05%,1.0417
2,original minus shuffled,18.00%,0.4244


Accuracy drop: 18.00 percentage points


In [24]:
# 3. State-shuffle effect by street.
street_rows = []
for street in ('preflop', 'flop', 'turn', 'river'):
    if street not in original_metrics or street not in shuffled_metrics:
        continue
    original = original_metrics[street]
    shuffled = shuffled_metrics[street]
    street_rows.append({
        'street': street,
        'n': int(original['n']),
        'original_accuracy': original['exact_accuracy'],
        'shuffled_state_accuracy': shuffled['exact_accuracy'],
        'accuracy_drop_pp': 100 * (original['exact_accuracy'] - shuffled['exact_accuracy']),
        'original_log_loss': original['log_loss'],
        'shuffled_state_log_loss': shuffled['log_loss'],
    })
street_shuffle_report = pd.DataFrame(street_rows)
display(street_shuffle_report.style.format({
    'original_accuracy': '{:.2%}',
    'shuffled_state_accuracy': '{:.2%}',
    'accuracy_drop_pp': '{:.2f}',
    'original_log_loss': '{:.4f}',
    'shuffled_state_log_loss': '{:.4f}',
}))

print(
    'Interpretation boundary: these results quantify sensitivity to the serialized state on this '
    'validation distribution. Candidate menus retain their own statistical signal, while shuffled '
    'state/menu pairings are semantically invalid and may be out of distribution. Therefore a small '
    'drop is not proof of absent poker understanding, and a large drop is not proof of poker reasoning.'
)


,street,n,original_accuracy,shuffled_state_accuracy,accuracy_drop_pp,original_log_loss,shuffled_state_log_loss
0,preflop,1027,85.00%,59.01%,26.00,0.3727,0.8968
1,flop,72,43.06%,31.94%,11.11,0.9962,1.4512
2,turn,402,60.95%,48.01%,12.94,0.9244,1.3051
3,river,499,58.52%,51.90%,6.61,0.8188,1.0688


Interpretation boundary: these results quantify sensitivity to the serialized state on this validation distribution. Candidate menus retain their own statistical signal, while shuffled state/menu pairings are semantically invalid and may be out of distribution. Therefore a small drop is not proof of absent poker understanding, and a large drop is not proof of poker reasoning.


In [25]:
# 4. Street x candidate-count analysis. Reuses only the original and shuffled predictions above.
# N is intentionally first: cells with low N are descriptive rather than reliable evidence.
street_order = ('preflop', 'flop', 'turn', 'river')
observed_candidate_counts = sorted({len(e.inputs.candidates) for e in validation_examples})
street_candidate_count_rows = []

for street in street_order:
    for candidate_count in observed_candidate_counts:
        indices = [
            i for i, example in enumerate(validation_examples)
            if example.street == street and len(example.inputs.candidates) == candidate_count
        ]
        if not indices:
            continue
        original_subset = [validation_examples[i] for i in indices]
        shuffled_subset = [shuffled_validation_examples[i] for i in indices]
        original_subset_metrics = evaluate(original_subset, original_lookup.__getitem__)
        shuffled_subset_metrics = evaluate(shuffled_subset, shuffled_lookup.__getitem__)
        street_candidate_count_rows.append({
            'street': street,
            'candidate_count': candidate_count,
            'n': len(indices),
            'original_accuracy': original_subset_metrics['exact_accuracy'],
            'shuffled_state_accuracy': shuffled_subset_metrics['exact_accuracy'],
            'accuracy_drop_pp': 100 * (
                original_subset_metrics['exact_accuracy'] - shuffled_subset_metrics['exact_accuracy']
            ),
        })

street_candidate_count_report = pd.DataFrame(street_candidate_count_rows)
street_candidate_count_report['street'] = pd.Categorical(
    street_candidate_count_report['street'], categories=street_order, ordered=True
)
street_candidate_count_report = street_candidate_count_report.sort_values(
    ['street', 'candidate_count'], ignore_index=True
)
display(street_candidate_count_report.style.format({
    'original_accuracy': '{:.2%}',
    'shuffled_state_accuracy': '{:.2%}',
    'accuracy_drop_pp': '{:.2f}',
}))

# Focus only on counts supported by at least 20 examples in at least two streets.
# This threshold is a display guardrail, not a statistical significance test.
FOCUSED_MIN_N = 20
eligible_counts = (
    street_candidate_count_report.loc[lambda frame: frame['n'] >= FOCUSED_MIN_N]
    .groupby('candidate_count', observed=True)['street']
    .nunique()
)
eligible_counts = eligible_counts[eligible_counts >= 2].index.tolist()
focused_street_candidate_count_report = street_candidate_count_report.loc[
    (street_candidate_count_report['candidate_count'].isin(eligible_counts))
    & (street_candidate_count_report['n'] >= FOCUSED_MIN_N)
].reset_index(drop=True)

if focused_street_candidate_count_report.empty:
    print(f'No candidate count has N >= {FOCUSED_MIN_N} in two or more streets.')
else:
    print(
        f'Focused cross-street comparison: candidate counts {eligible_counts} '
        f'with N >= {FOCUSED_MIN_N} in at least two streets.'
    )
    display(focused_street_candidate_count_report.style.format({
        'original_accuracy': '{:.2%}',
        'shuffled_state_accuracy': '{:.2%}',
        'accuracy_drop_pp': '{:.2f}',
    }))


,street,candidate_count,n,original_accuracy,shuffled_state_accuracy,accuracy_drop_pp
0,preflop,1,97,100.00%,100.00%,0.00
1,preflop,2,748,85.70%,58.02%,27.67
2,preflop,3,137,75.91%,43.80%,32.12
3,preflop,4,45,68.89%,33.33%,35.56
4,flop,2,13,38.46%,46.15%,-7.69
5,flop,3,48,41.67%,31.25%,10.42
6,flop,4,11,54.55%,18.18%,36.36
7,turn,2,57,73.68%,56.14%,17.54
8,turn,3,294,61.56%,47.62%,13.95
9,turn,4,51,43.14%,41.18%,1.96


Focused cross-street comparison: candidate counts [2, 3, 4] with N >= 20 in at least two streets.


,street,candidate_count,n,original_accuracy,shuffled_state_accuracy,accuracy_drop_pp
0,preflop,2,748,85.70%,58.02%,27.67
1,preflop,3,137,75.91%,43.80%,32.12
2,preflop,4,45,68.89%,33.33%,35.56
3,flop,3,48,41.67%,31.25%,10.42
4,turn,2,57,73.68%,56.14%,17.54
5,turn,3,294,61.56%,47.62%,13.95
6,turn,4,51,43.14%,41.18%,1.96
7,river,2,275,67.27%,62.91%,4.36
8,river,3,193,48.70%,39.38%,9.33
9,river,4,31,41.94%,32.26%,9.68


## Independent Analysis: Candidate Menu Composition

This follows the fixed original and state-shuffled evaluations above. It does not retrain, alter the validation split, create predictions, or change the shuffle. The tables describe whether street-level shuffle differences persist within comparable candidate-menu compositions; low-N cells are descriptive only.


In [26]:
# Candidate menu representation: PokerBench candidates are normalized strings such as
# 'fold', 'call', 'check', 'bet 16bb', and 'raise 29bb'.
# The helpers below preserve candidate order and only normalize superficial text formatting.
from collections import Counter

import numpy as np

MENU_FOCUSED_MIN_N = 20  # Display guardrail; this is not a significance threshold.
MENU_STREET_ORDER = ('preflop', 'flop', 'turn', 'river')

def normalize_candidate_text(candidate):
    """Case/whitespace normalization while retaining action sizing."""
    return ' '.join(str(candidate).casefold().split())


def candidate_action_type(candidate):
    """Transparent action-type normalization: first PokerBench action token only."""
    normalized = normalize_candidate_text(candidate)
    if not normalized:
        raise ValueError('Empty candidate text')
    return normalized.split()[0]


def action_type_signature(candidates):
    return tuple(candidate_action_type(candidate) for candidate in candidates)


def exact_normalized_signature(candidates):
    return tuple(normalize_candidate_text(candidate) for candidate in candidates)


candidate_menu_frame = pd.DataFrame([
    {
        'validation_index': index,
        'street': example.street,
        'candidate_count': len(example.inputs.candidates),
        'action_type_signature': action_type_signature(example.inputs.candidates),
        'exact_normalized_signature': exact_normalized_signature(example.inputs.candidates),
        'target_action': normalize_candidate_text(example.target_action),
    }
    for index, example in enumerate(validation_examples)
])
candidate_menu_frame['street'] = pd.Categorical(
    candidate_menu_frame['street'], categories=MENU_STREET_ORDER, ordered=True
)
assert len(candidate_menu_frame) == len(validation_examples)
assert candidate_menu_frame['candidate_count'].eq(
    [len(example.inputs.candidates) for example in validation_examples]
).all()


def menu_composition_table(signature_column):
    """Count ordered menu signatures and their share of every street's validation examples."""
    table = (
        candidate_menu_frame
        .groupby(['street', 'candidate_count', signature_column], observed=True)
        .size()
        .rename('n')
        .reset_index()
        .rename(columns={signature_column: 'menu_signature'})
    )
    table['pct_within_street'] = table['n'] / table.groupby('street', observed=True)['n'].transform('sum')
    return table.sort_values(
        ['street', 'n', 'candidate_count', 'menu_signature'],
        ascending=[True, False, True, True],
        ignore_index=True,
    )


action_type_menu_composition = menu_composition_table('action_type_signature')
exact_menu_composition = menu_composition_table('exact_normalized_signature')
print('Action-type signatures (sizing ignored; candidate order retained)')
display(action_type_menu_composition.style.format({'pct_within_street': '{:.1%}'}))
print('Exact normalized signatures (sizing retained; candidate order retained)')
display(exact_menu_composition.style.format({'pct_within_street': '{:.1%}'}))


Action-type signatures (sizing ignored; candidate order retained)


,street,candidate_count,menu_signature,n,pct_within_street
0,preflop,2,"('call', 'fold')",569,55.4%
1,preflop,2,"('raise', 'fold')",103,10.0%
2,preflop,1,"('fold',)",97,9.4%
3,preflop,2,"('all-in', 'fold')",76,7.4%
4,preflop,3,"('all-in', 'call', 'fold')",38,3.7%
5,preflop,3,"('raise', 'call', 'fold')",33,3.2%
6,preflop,3,"('call', 'raise', 'fold')",28,2.7%
7,preflop,3,"('raise', 'all-in', 'fold')",27,2.6%
8,preflop,4,"('raise', 'all-in', 'call', 'fold')",17,1.7%
9,preflop,4,"('all-in', 'raise', 'call', 'fold')",13,1.3%


Exact normalized signatures (sizing retained; candidate order retained)


,street,candidate_count,menu_signature,n,pct_within_street
0,preflop,2,"('call', 'fold')",569,55.4%
1,preflop,1,"('fold',)",97,9.4%
2,preflop,2,"('all-in', 'fold')",76,7.4%
3,preflop,3,"('all-in', 'call', 'fold')",38,3.7%
4,preflop,2,"('raise 13bb', 'fold')",12,1.2%
5,preflop,2,"('raise 15bb', 'fold')",12,1.2%
6,preflop,2,"('raise 14bb', 'fold')",11,1.1%
7,preflop,3,"('call', 'raise 10bb', 'fold')",11,1.1%
8,preflop,2,"('raise 2.5bb', 'fold')",10,1.0%
9,preflop,2,"('raise 6.5bb', 'fold')",10,1.0%


In [27]:
# Reuse cached probabilities through the existing evaluator; no model call or fitting occurs here.
def format_target_distribution(target_actions):
    counts = Counter(target_actions)
    total = sum(counts.values())
    return '; '.join(
        f'{action}: {count} ({count / total:.0%})'
        for action, count in sorted(counts.items())
    )


def menu_shuffle_sensitivity(signature_column):
    rows = []
    grouped = candidate_menu_frame.groupby(
        ['street', 'candidate_count', signature_column], observed=True, sort=False
    )
    for (street, candidate_count, signature), group in grouped:
        indices = group['validation_index'].tolist()
        original_subset = [validation_examples[index] for index in indices]
        shuffled_subset = [shuffled_validation_examples[index] for index in indices]
        original_metrics_for_menu = evaluate(original_subset, original_lookup.__getitem__)
        shuffled_metrics_for_menu = evaluate(shuffled_subset, shuffled_lookup.__getitem__)
        rows.append({
            'street': street,
            'candidate_count': candidate_count,
            'menu_signature': signature,
            'n': len(indices),
            'target_action_distribution': format_target_distribution(group['target_action']),
            'original_accuracy': original_metrics_for_menu['exact_accuracy'],
            'shuffled_state_accuracy': shuffled_metrics_for_menu['exact_accuracy'],
            'accuracy_drop_pp': 100 * (
                original_metrics_for_menu['exact_accuracy']
                - shuffled_metrics_for_menu['exact_accuracy']
            ),
            'reliability_note': (
                f'focused (N >= {MENU_FOCUSED_MIN_N})'
                if len(indices) >= MENU_FOCUSED_MIN_N
                else f'descriptive (N < {MENU_FOCUSED_MIN_N})'
            ),
        })
    return pd.DataFrame(rows).sort_values(
        ['street', 'candidate_count', 'n', 'menu_signature'],
        ascending=[True, True, False, True],
        ignore_index=True,
    )


action_type_menu_sensitivity = menu_shuffle_sensitivity('action_type_signature')
exact_menu_sensitivity = menu_shuffle_sensitivity('exact_normalized_signature')

print('State-shuffle sensitivity within action-type signatures')
display(action_type_menu_sensitivity.style.format({
    'original_accuracy': '{:.2%}',
    'shuffled_state_accuracy': '{:.2%}',
    'accuracy_drop_pp': '{:.2f}',
}))
print('State-shuffle sensitivity within exact normalized signatures')
display(exact_menu_sensitivity.style.format({
    'original_accuracy': '{:.2%}',
    'shuffled_state_accuracy': '{:.2%}',
    'accuracy_drop_pp': '{:.2f}',
}))


State-shuffle sensitivity within action-type signatures


,street,candidate_count,menu_signature,n,target_action_distribution,original_accuracy,shuffled_state_accuracy,accuracy_drop_pp,reliability_note
0,flop,2,"('check', 'bet')",13,bet 1bb: 1 (8%); bet 2bb: 7 (54%); bet 3bb: 1 (8%); bet 4bb: 1 (8%); check: 3 (23%),38.46%,46.15%,-7.69,descriptive (N < 20)
1,flop,3,"('fold', 'call', 'raise')",44,call: 33 (75%); fold: 5 (11%); raise 10bb: 1 (2%); raise 16bb: 1 (2%); raise 19bb: 1 (2%); raise 31bb: 1 (2%); raise 4bb: 1 (2%); raise 7bb: 1 (2%),40.91%,27.27%,13.64,focused (N >= 20)
2,flop,3,"('check', 'bet', 'bet')",4,bet 10bb: 1 (25%); check: 3 (75%),50.00%,75.00%,-25.00,descriptive (N < 20)
3,flop,4,"('fold', 'call', 'raise', 'raise')",11,call: 7 (64%); fold: 1 (9%); raise 32bb: 1 (9%); raise 45bb: 1 (9%); raise 87bb: 1 (9%),54.55%,18.18%,36.36,descriptive (N < 20)
4,preflop,1,"('fold',)",97,fold: 97 (100%),100.00%,100.00%,0.00,focused (N >= 20)
5,preflop,2,"('call', 'fold')",569,call: 340 (60%); fold: 229 (40%),83.48%,52.55%,30.93,focused (N >= 20)
6,preflop,2,"('raise', 'fold')",103,fold: 27 (26%); raise 11bb: 8 (8%); raise 12.5bb: 1 (1%); raise 12bb: 4 (4%); raise 13.1bb: 1 (1%); raise 13.5bb: 4 (4%); raise 13bb: 9 (9%); raise 14.5bb: 2 (2%); raise 14bb: 8 (8%); raise 15bb: 6 (6%); raise 16bb: 2 (2%); raise 2.3bb: 4 (4%); raise 2.5bb: 10 (10%); raise 2bb: 7 (7%); raise 38bb: 2 (2%); raise 6.5bb: 5 (5%); raise 9.5bb: 3 (3%),87.38%,57.28%,30.10,focused (N >= 20)
7,preflop,2,"('all-in', 'fold')",76,fold: 76 (100%),100.00%,100.00%,0.00,focused (N >= 20)
8,preflop,3,"('all-in', 'call', 'fold')",38,call: 19 (50%); fold: 19 (50%),71.05%,50.00%,21.05,focused (N >= 20)
9,preflop,3,"('raise', 'call', 'fold')",33,call: 12 (36%); fold: 4 (12%); raise 11.5bb: 2 (6%); raise 21bb: 2 (6%); raise 23bb: 1 (3%); raise 3bb: 8 (24%); raise 7.5bb: 4 (12%),54.55%,39.39%,15.15,focused (N >= 20)


State-shuffle sensitivity within exact normalized signatures


,street,candidate_count,menu_signature,n,target_action_distribution,original_accuracy,shuffled_state_accuracy,accuracy_drop_pp,reliability_note
0,flop,2,"('check', 'bet 2bb')",8,bet 2bb: 7 (88%); check: 1 (12%),37.50%,25.00%,12.50,descriptive (N < 20)
1,flop,2,"('check', 'bet 4bb')",2,bet 4bb: 1 (50%); check: 1 (50%),50.00%,100.00%,-50.00,descriptive (N < 20)
2,flop,2,"('check', 'bet 1bb')",1,bet 1bb: 1 (100%),0.00%,0.00%,0.00,descriptive (N < 20)
3,flop,2,"('check', 'bet 3bb')",1,bet 3bb: 1 (100%),0.00%,100.00%,-100.00,descriptive (N < 20)
4,flop,2,"('check', 'bet 5bb')",1,check: 1 (100%),100.00%,100.00%,0.00,descriptive (N < 20)
5,flop,3,"('fold', 'call', 'raise 13bb')",6,call: 6 (100%),66.67%,50.00%,16.67,descriptive (N < 20)
6,flop,3,"('fold', 'call', 'raise 26bb')",5,call: 4 (80%); fold: 1 (20%),20.00%,20.00%,0.00,descriptive (N < 20)
7,flop,3,"('fold', 'call', 'raise 28bb')",5,call: 4 (80%); fold: 1 (20%),60.00%,20.00%,40.00,descriptive (N < 20)
8,flop,3,"('fold', 'call', 'raise 7bb')",5,call: 3 (60%); fold: 1 (20%); raise 7bb: 1 (20%),40.00%,20.00%,20.00,descriptive (N < 20)
9,flop,3,"('fold', 'call', 'raise 10bb')",3,call: 2 (67%); raise 10bb: 1 (33%),0.00%,33.33%,-33.33,descriptive (N < 20)


In [28]:
# Explicit preflop-versus-river comparison for the same ordered menu signature.
def shared_preflop_river_menus(sensitivity_table):
    focused = sensitivity_table.loc[
        (sensitivity_table['street'].isin(['preflop', 'river']))
        & (sensitivity_table['n'] >= MENU_FOCUSED_MIN_N)
    ].copy()
    key_columns = ['candidate_count', 'menu_signature']
    common_keys = (
        focused.groupby(key_columns, dropna=False)['street']
        .agg(lambda streets: set(streets) == {'preflop', 'river'})
    )
    common_keys = common_keys[common_keys].index
    if len(common_keys) == 0:
        return focused.iloc[0:0].copy()
    common_index = pd.MultiIndex.from_tuples(common_keys, names=key_columns)
    return focused.set_index(key_columns).loc[common_index].reset_index().sort_values(
        ['candidate_count', 'menu_signature', 'street'], ignore_index=True
    )


common_action_type_preflop_river = shared_preflop_river_menus(action_type_menu_sensitivity)
common_exact_preflop_river = shared_preflop_river_menus(exact_menu_sensitivity)

for label, comparison in [
    ('Action-type signatures shared by preflop and river', common_action_type_preflop_river),
    ('Exact normalized signatures shared by preflop and river', common_exact_preflop_river),
]:
    print(label)
    if comparison.empty:
        print(f'No shared signature has N >= {MENU_FOCUSED_MIN_N} in both streets.')
    else:
        display(comparison.style.format({
            'original_accuracy': '{:.2%}',
            'shuffled_state_accuracy': '{:.2%}',
            'accuracy_drop_pp': '{:.2f}',
        }))

# Make the requested two- and three-candidate comparison easy to inspect when supported.
common_two_three = common_action_type_preflop_river.loc[
    common_action_type_preflop_river['candidate_count'].isin([2, 3])
]
print('Focused 2- and 3-candidate action-type comparison')
if common_two_three.empty:
    print(f'No 2- or 3-candidate action-type signature meets N >= {MENU_FOCUSED_MIN_N} in both streets.')
else:
    display(common_two_three.style.format({
        'original_accuracy': '{:.2%}',
        'shuffled_state_accuracy': '{:.2%}',
        'accuracy_drop_pp': '{:.2f}',
    }))


def total_variation_distance(composition, left_street, right_street):
    left = composition.loc[composition['street'] == left_street].set_index(
        ['candidate_count', 'menu_signature']
    )['pct_within_street']
    right = composition.loc[composition['street'] == right_street].set_index(
        ['candidate_count', 'menu_signature']
    )['pct_within_street']
    all_keys = left.index.union(right.index)
    return 0.5 * (left.reindex(all_keys, fill_value=0) - right.reindex(all_keys, fill_value=0)).abs().sum()


def weighted_drop_by_street(shared_table):
    if shared_table.empty:
        return {}
    return {
        street: np.average(rows['accuracy_drop_pp'], weights=rows['n'])
        for street, rows in shared_table.groupby('street', observed=True)
    }


action_type_tvd = total_variation_distance(action_type_menu_composition, 'preflop', 'river')
exact_tvd = total_variation_distance(exact_menu_composition, 'preflop', 'river')
common_action_type_drops = weighted_drop_by_street(common_action_type_preflop_river)

print('Factual summary (descriptive; no causal or poker-understanding claim):')
print(
    f'1. Preflop-vs-river menu-composition total-variation distance: '
    f'{action_type_tvd:.3f} for action types and {exact_tvd:.3f} for exact normalized menus '
    f'(0 means identical composition; 1 means no overlap).'
)
river_focused = action_type_menu_sensitivity.loc[
    (action_type_menu_sensitivity['street'] == 'river')
    & (action_type_menu_sensitivity['n'] >= MENU_FOCUSED_MIN_N)
]
if river_focused.empty:
    print(f'2. No river action-type menu has N >= {MENU_FOCUSED_MIN_N}; do not assess post-shuffle menu signal from small cells.')
else:
    strongest_river = river_focused.sort_values('shuffled_state_accuracy', ascending=False).iloc[0]
    print(
        '2. Highest shuffled-state river accuracy among supported action-type menus: '
        f"{strongest_river['shuffled_state_accuracy']:.2%} for {strongest_river['menu_signature']} "
        f"(N={strongest_river['n']}, target distribution: {strongest_river['target_action_distribution']})."
    )
if {'preflop', 'river'} <= set(common_action_type_drops):
    print(
        '3. Within action-type signatures supported in both streets, N-weighted accuracy drops are '
        f"{common_action_type_drops['preflop']:.2f} pp (preflop) and "
        f"{common_action_type_drops['river']:.2f} pp (river). Inspect the displayed signature rows "
        'before judging whether a gap remains.'
    )
else:
    print('3. No action-type signature has sufficient N in both preflop and river, so this control cannot resolve the street gap.')
print(
    '4. Main remaining confound: shuffled state/menu pairs are semantically mismatched and potentially '
    'out of distribution; target-action imbalance and unmeasured menu/state correlations can also remain. '
    'A shuffle drop is sensitivity to correct serialized state, not an additive share of model performance.'
)


Action-type signatures shared by preflop and river
No shared signature has N >= 20 in both streets.
Exact normalized signatures shared by preflop and river
No shared signature has N >= 20 in both streets.
Focused 2- and 3-candidate action-type comparison
No 2- or 3-candidate action-type signature meets N >= 20 in both streets.
Factual summary (descriptive; no causal or poker-understanding claim):
1. Preflop-vs-river menu-composition total-variation distance: 1.000 for action types and 1.000 for exact normalized menus (0 means identical composition; 1 means no overlap).
2. Highest shuffled-state river accuracy among supported action-type menus: 64.25% for ('check', 'bet') (N=207, target distribution: bet 10bb: 4 (2%); bet 11bb: 4 (2%); bet 12bb: 2 (1%); bet 13bb: 2 (1%); bet 17bb: 4 (2%); bet 18bb: 4 (2%); bet 23bb: 9 (4%); bet 24bb: 3 (1%); bet 25bb: 3 (1%); bet 27bb: 3 (1%); bet 29bb: 11 (5%); bet 34bb: 1 (0%); bet 3bb: 2 (1%); bet 4bb: 4 (2%); bet 5bb: 1 (0%); bet 7bb: 2 (1%); bet 8b

## Independent Analysis: Repeated State-Shuffle Robustness

This robustness check repeats the existing fixed-point-free state shuffle over 30 deterministic seeds. It reuses the saved BERT Mini checkpoint, the unchanged validation set, and the existing serialization, prediction, and evaluation pipeline; it does not train or fine-tune a model.


In [29]:
# Repeat exactly the existing state-shuffle intervention with different deterministic seeds.
# Every shuffle is a derangement: a recipient never receives its own state index.
ROBUSTNESS_SEEDS = tuple(range(30))
ROBUSTNESS_STREETS = ('preflop', 'flop', 'turn', 'river')

def state_shuffle_for_seed(seed):
    donors = list(range(len(validation_examples)))
    rng = random.Random(seed)
    while True:
        rng.shuffle(donors)
        if all(recipient != donor for recipient, donor in enumerate(donors)):
            break
    shuffled_examples = [
        DecisionExample(
            source_id=example.source_id,
            street=example.street,
            inputs=DecisionInputs(
                context=validation_examples[donor].inputs.context,
                candidates=example.inputs.candidates,
            ),
            target_index=example.target_index,
        )
        for example, donor in zip(validation_examples, donors, strict=True)
    ]
    # Preserve the original intervention's identity/menu/label guardrails for every seed.
    assert sorted(donors) == list(range(len(validation_examples)))
    assert all(recipient != donor for recipient, donor in enumerate(donors))
    assert [e.source_id for e in shuffled_examples] == [e.source_id for e in validation_examples]
    assert [e.street for e in shuffled_examples] == [e.street for e in validation_examples]
    assert [e.inputs.candidates for e in shuffled_examples] == [e.inputs.candidates for e in validation_examples]
    assert [e.target_index for e in shuffled_examples] == [e.target_index for e in validation_examples]
    return shuffled_examples

seed_rows = []
seed_street_rows = []
for seed in ROBUSTNESS_SEEDS:
    repeated_shuffled_examples = state_shuffle_for_seed(seed)
    repeated_encoded = encode_examples(
        repeated_shuffled_examples, analysis_tokenizer, max_length=analysis_max_length
    )
    repeated_probabilities = predict_encoded(
        analysis_model, repeated_encoded, pad_token_id=analysis_pad
    )
    repeated_lookup = prediction_lookup(repeated_shuffled_examples, repeated_probabilities)
    repeated_metrics = evaluate_by_street(
        repeated_shuffled_examples, repeated_lookup.__getitem__
    )
    overall = repeated_metrics['overall']
    seed_rows.append({
        'seed': seed,
        'shuffled_state_accuracy': overall['exact_accuracy'],
        'accuracy_drop_pp': 100 * (
            original_metrics['overall']['exact_accuracy'] - overall['exact_accuracy']
        ),
    })
    for street in ROBUSTNESS_STREETS:
        if street not in repeated_metrics:
            continue
        metrics = repeated_metrics[street]
        seed_street_rows.append({
            'seed': seed,
            'street': street,
            'n': int(metrics['n']),
            'shuffled_state_accuracy': metrics['exact_accuracy'],
            'accuracy_drop_pp': 100 * (
                original_metrics[street]['exact_accuracy'] - metrics['exact_accuracy']
            ),
        })

repeated_shuffle_by_seed = pd.DataFrame(seed_rows).sort_values('seed', ignore_index=True)
repeated_shuffle_by_street_seed = pd.DataFrame(seed_street_rows)
display(repeated_shuffle_by_seed.style.format({
    'shuffled_state_accuracy': '{:.2%}',
    'accuracy_drop_pp': '{:.2f}',
}))

summary_rows = [{
    'subset': 'overall',
    'n': int(original_metrics['overall']['n']),
    'original_accuracy': original_metrics['overall']['exact_accuracy'],
    'mean_shuffled_accuracy': repeated_shuffle_by_seed['shuffled_state_accuracy'].mean(),
    'shuffled_accuracy_std': repeated_shuffle_by_seed['shuffled_state_accuracy'].std(ddof=1),
    'mean_accuracy_drop_pp': repeated_shuffle_by_seed['accuracy_drop_pp'].mean(),
    'accuracy_drop_std_pp': repeated_shuffle_by_seed['accuracy_drop_pp'].std(ddof=1),
}]
for street in ROBUSTNESS_STREETS:
    street_runs = repeated_shuffle_by_street_seed.loc[
        repeated_shuffle_by_street_seed['street'] == street
    ]
    if street_runs.empty:
        continue
    summary_rows.append({
        'subset': street,
        'n': int(original_metrics[street]['n']),
        'original_accuracy': original_metrics[street]['exact_accuracy'],
        'mean_shuffled_accuracy': street_runs['shuffled_state_accuracy'].mean(),
        'shuffled_accuracy_std': street_runs['shuffled_state_accuracy'].std(ddof=1),
        'mean_accuracy_drop_pp': street_runs['accuracy_drop_pp'].mean(),
        'accuracy_drop_std_pp': street_runs['accuracy_drop_pp'].std(ddof=1),
    })
repeated_shuffle_summary = pd.DataFrame(summary_rows)
display(repeated_shuffle_summary.style.format({
    'original_accuracy': '{:.2%}',
    'mean_shuffled_accuracy': '{:.2%}',
    'shuffled_accuracy_std': '{:.2%}',
    'mean_accuracy_drop_pp': '{:.2f}',
    'accuracy_drop_std_pp': '{:.2f}',
}))

print(
    f"Across {len(ROBUSTNESS_SEEDS)} state-shuffle seeds, overall shuffled accuracy is "
    f"{repeated_shuffle_by_seed['shuffled_state_accuracy'].mean():.2%} "
    f"± {repeated_shuffle_by_seed['shuffled_state_accuracy'].std(ddof=1):.2%}, and the mean "
    f"drop from the fixed original accuracy is {repeated_shuffle_by_seed['accuracy_drop_pp'].mean():.2f} "
    f"± {repeated_shuffle_by_seed['accuracy_drop_pp'].std(ddof=1):.2f} pp. "
    "This quantifies sensitivity to the particular state permutation; shuffled state/menu pairs remain "
    "semantically mismatched and may be out of distribution."
)


,seed,shuffled_state_accuracy,accuracy_drop_pp
0,0,53.85%,18.20
1,1,53.20%,18.85
2,2,52.75%,19.30
3,3,52.85%,19.20
4,4,52.10%,19.95
5,5,53.00%,19.05
6,6,53.70%,18.35
7,7,53.50%,18.55
8,8,53.65%,18.40
9,9,53.25%,18.80


,subset,n,original_accuracy,mean_shuffled_accuracy,shuffled_accuracy_std,mean_accuracy_drop_pp,accuracy_drop_std_pp
0,overall,2000,72.05%,53.33%,0.63%,18.72,0.63
1,preflop,1027,85.00%,58.45%,1.11%,26.55,1.11
2,flop,72,43.06%,38.52%,4.54%,4.54,4.54
3,turn,402,60.95%,45.95%,1.87%,15.00,1.87
4,river,499,58.52%,50.86%,1.50%,7.66,1.50


Across 30 state-shuffle seeds, overall shuffled accuracy is 53.33% ± 0.63%, and the mean drop from the fixed original accuracy is 18.72 ± 0.63 pp. This quantifies sensitivity to the particular state permutation; shuffled state/menu pairs remain semantically mismatched and may be out of distribution.
